# 🏥 CareSphere: Qwen2.5-0.5B Instruction Fine-Tuning Workflow
### Strictly Dataset-Grounded Healthcare Assistant for Frontline ASHA/ANM Workers

This reproducible Jupyter Notebook fine-tunes **Qwen2.5-0.5B-Instruct** (or 1.5B) on the official Chandrapuri Sub-Centre healthcare protocol dataset.

**Key Objectives:**
1. Ground the model strictly in the 9 Government of India protocol documents (107 leaf sections).
2. Enforce hard-coded citations (e.g. `ANC-1.2`, `IMM-2.1`, `ESC-3.1`) in every factual response.
3. Train safe refusals (`SCP-1.2`) for unapproved medicines, unlisted conditions, and non-health domains.
4. Export merged weights into **GGUF (q4_k_m)** format for 100% offline deployment with **Ollama**.

**Hardware Requirement:** Free Google Colab T4 GPU (or any CUDA GPU with >= 6GB VRAM).

## Step 1: Install Dependencies
Install Hugging Face libraries, PEFT (LoRA), TRL, and 4-bit quantization utilities.

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets
!pip install -q sentencepiece protobuf

## Step 2: Verify GPU Hardware Environment

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
!nvidia-smi

## Step 3: Load the Verified CareSphere Dataset
Ensure `caresphere_sft_dataset.jsonl` is present in your working directory.

In [ ]:
from datasets import load_dataset
import json

# Load training dataset
dataset_path = 'caresphere_sft_dataset.jsonl'
dataset = load_dataset('json', data_files=dataset_path, split='train')

# Split 90% train, 10% validation
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
val_data = split_dataset['test']

print(f'Train samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print('\nSample Example:')
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

## Step 4: Configure QLoRA (4-bit Quantization)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

print('Loading base model...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 5: Train with SFTTrainer
We format the examples using Qwen's ChatML template and train for 3 epochs.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

def format_chatml(batch):
    texts = []
    for msgs in batch['messages']:
        formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(formatted)
    return texts

training_args = TrainingArguments(
    output_dir='caresphere_lora_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    peft_config=peft_config,
    dataset_text_field='text',
    formatting_func=format_chatml,
    max_seq_length=512,
    tokenizer=tokenizer
)

print('Starting fine-tuning...')
trainer.train()

# Save LoRA adapter
ADAPTER_PATH = 'caresphere_qwen_adapter'
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'LoRA adapter saved to: {ADAPTER_PATH}')

## Step 6: Merge Weights and Export GGUF for Ollama

In [ ]:
from peft import AutoPeftModelForCausalLM

print('Merging LoRA adapter with base model...')
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_PATH,
    device_map='cpu',
    torch_dtype=torch.float16
)
merged = merged_model.merge_and_unload()

MERGED_DIR = 'caresphere_qwen_merged'
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved to: {MERGED_DIR}')

# Convert to GGUF using llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py caresphere_qwen_merged --outtype q4_k_m --outfile caresphere_qwen_q4km.gguf
print('Successfully converted to GGUF: caresphere_qwen_q4km.gguf')

## Step 7: Create Ollama Modelfile
Use the generated GGUF file to create your offline Ollama model on your local laptop.

In [ ]:
modelfile_content = '''FROM ./caresphere_qwen_q4km.gguf
PARAMETER temperature 0.1
PARAMETER top_p 0.8
PARAMETER stop "<|im_end|>"

SYSTEM """You are CareSphere, an offline clinical decision support assistant for frontline health workers (ASHA/ANM) in Chandrapuri Sub-Centre. Answer strictly from official protocols. Cite leaf sections (e.g. ANC-1.2). Refuse unsupported questions (SCP-1.2)."""
'''

with open('Modelfile.caresphere', 'w') as f: 
    f.write(modelfile_content)

print('Modelfile.caresphere created.')
print('\nOn your local machine with Ollama installed, run:')
print('  ollama create caresphere-qwen -f Modelfile.caresphere')